**Ingest Constructors - Incremental**

Reads `constructors.json` from the batch landing folder, adds metadata, and writes to `formula1_incr.bronze.constructors` partitioned by `batch_id`.

**Load config and helpers**

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze_helpers

In [0]:
dbutils.widgets.text('p_batch_id','')
batch_id= dbutils.widgets.get('p_batch_id')

**Verify written data**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as f

In [0]:
source_file = f'{loding_folder_path}/{batch_id}/constructors.json' # to replace the load in read api
table_name = f'{catalog_name}.{bronze_schema}.constructors'

In [0]:
source_file

**Define schema** (DDL string format)

In [0]:
from pyspark.sql.types import *
constructor_schema = 'constructorId STRING, name STRING, nationality STRING, url STRING'

**Read JSON file**

In [0]:
constructors_df = (
    spark.read
    .format('json')
    .option('Headers',True)
    # .option('inferSchema', True)
    .schema(constructor_schema)
    .load(source_file)
)
display(constructors_df)


**Add metadata columns**

In [0]:
constructors_final_df = add_ingestion_metadata(constructors_df)


**Write to bronze Delta table** (overwrite per batch partition)

In [0]:
write_to_bronze(
    input_df = constructors_final_df,
    table_name = table_name,
    batch_id = batch_id
)

In [0]:
display(spark.table(table_name))